# YOLO Fine-tuning for Road Damage Detection

Essential code to train YOLO model for road damage detection:
- Pothole, Alligator Crack, Transverse Crack, Longitudinal Crack

In [1]:
# Install and import required packages
%pip install ultralytics -q

import os
import yaml
import shutil
import random
from glob import glob
from ultralytics import YOLO

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load YOLO11m model
model = YOLO('yolo11m.pt')

100%|██████████| 38.8M/38.8M [00:03<00:00, 10.5MB/s]


In [3]:
# Prepare dataset for YOLO training
def prepare_yolo_dataset(data_dir="data", output_dir="yolo_dataset"):
    # Create directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    # Collect all image-label pairs from country directories
    all_pairs = []
    country_dirs = [d for d in os.listdir(data_dir) if d.startswith('country_')]
    
    for country_dir in country_dirs:
        images_path = f"{data_dir}/{country_dir}/images"
        labels_path = f"{data_dir}/{country_dir}/labels"
        
        for img_file in glob(f"{images_path}/*.jpg") + glob(f"{images_path}/*.png"):
            base_name = os.path.splitext(os.path.basename(img_file))[0]
            label_file = f"{labels_path}/{base_name}.txt"
            if os.path.exists(label_file):
                all_pairs.append((img_file, label_file))
    
    # Split dataset (70% train, 15% val, 15% test)
    random.seed(42)
    random.shuffle(all_pairs)
    
    n = len(all_pairs)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)
    
    splits = {
        'train': all_pairs[:train_end],
        'val': all_pairs[train_end:val_end],
        'test': all_pairs[val_end:]
    }
    
    # Copy files to respective directories
    for split_name, pairs in splits.items():
        for img_path, lbl_path in pairs:
            shutil.copy(img_path, f"{output_dir}/{split_name}/images/{os.path.basename(img_path)}")
            shutil.copy(lbl_path, f"{output_dir}/{split_name}/labels/{os.path.basename(lbl_path)}")
    
    # Create YAML configuration
    yaml_data = {
        'path': os.path.abspath(output_dir),
        'train': 'train/images',
        'val': 'val/images', 
        'test': 'test/images',
        'nc': 4,
        'names': ['Pothole', 'Alligator Crack', 'Transverse Crack', 'Longitudinal Crack']
    }
    
    with open(f"{output_dir}/data.yaml", 'w') as f:
        yaml.dump(yaml_data, f, sort_keys=False)
    
    return output_dir

# Prepare dataset if not exists
if not os.path.exists('yolo_dataset/data.yaml'):
    prepare_yolo_dataset()

In [4]:
# Train YOLO model
results = model.train(
    data='yolo_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    project='runs',
    name='road_damage_detection'
)

Ultralytics 8.3.169 🚀 Python-3.12.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=road_damage_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=Tr

100%|██████████| 5.35M/5.35M [00:00<00:00, 11.8MB/s]


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2736.1±1232.3 MB/s, size: 87.9 KB)


train: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/train/labels... 4227 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4227/4227 [00:03<00:00, 1162.19it/s]


train: New cache created: /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/train/labels.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1815.8±814.8 MB/s, size: 68.4 KB)


...

                   all        906       2454      0.436      0.405      0.364      0.161



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      17/50        11G      1.845      2.021      1.736         12        640: 100%|██████████| 265/265 [00:48<00:00,  5.47it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 29/29 [00:04<00:00,  6.10it/s]

                   all        906       2454      0.465      0.409      0.391      0.172


KeyboardInterrupt: 

In [ ]:
# Evaluate model
results = model.val(data='yolo_dataset/data.yaml', split='test')

# Print metrics
print(f"mAP@0.5: {results.box.map50:.4f}")
print(f"mAP@0.5:0.95: {results.box.map:.4f}")
print(f"Precision: {results.box.mp:.4f}")
print(f"Recall: {results.box.mr:.4f}")

# Per-class F1 scores
for i, name in enumerate(results.names.values()):
    if i < len(results.box.f1):
        print(f"{name}: F1={results.box.f1[i]:.3f}")

In [ ]:
# Save best model
train_dirs = sorted(glob('runs/detect/road_damage_detection*'), key=os.path.getmtime, reverse=True)
if train_dirs:
    best_model_path = f"{train_dirs[0]}/weights/best.pt"
    os.makedirs('yolo_checkpoints', exist_ok=True)
    shutil.copy(best_model_path, 'yolo_checkpoints/best_road_damage.pt')
    print(f"Model saved: yolo_checkpoints/best_road_damage.pt")

In [ ]:
# Load saved model for inference
# trained_model = YOLO('yolo_checkpoints/best_road_damage.pt')
# results = trained_model('path/to/image.jpg')
# results[0].show()